# Hướng dẫn Huấn luyện LoRA trên Google Colab Free (GPU T4)
*Dự án: GPT-Image Bone Age Synthesis*

Notebook này hỗ trợ 2 cách tiếp cận để thiết lập dự án trên Google Colab:
* **Cách 1 (Khuyên dùng - Nhanh & Sạch):** Clone code từ Git của bạn và tải dữ liệu trực tiếp từ Kaggle API (chỉ mất ~1 phút tải dữ liệu).
* **Cách 2:** Đồng bộ toàn bộ code qua Google Drive.

--- (BỎ QUA NẾU BẠN CHỌN CÁCH 2) ---
## CÁCH 1: CLONE CODE TỪ GIT & TẢI DỮ LIỆU QUA KAGGLE API

### Bước 1.1: Clone dự án từ kho Git cá nhân của bạn

In [ ]:
# Thay thế URL dưới đây bằng kho Git chứa mã nguồn của bạn
!git clone https://github.com/YOUR_USERNAME/gpt-image-bone-age-synthesis.git
%cd gpt-image-bone-age-synthesis

### Bước 1.2: Tự động tải bộ dữ liệu RSNA từ Kaggle về Colab
Điền `username` và `key` lấy từ file `kaggle.json` trên tài khoản Kaggle của bạn vào ô dưới đây để tự động tải.

In [ ]:
import json
import os

# ĐIỀN THÔNG TIN KAGGLE CỦA BẠN VÀO ĐÂY:
kaggle_creds = {
    "username": "YOUR_KAGGLE_USERNAME",
    "key": "YOUR_KAGGLE_KEY"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

!chmod 600 /root/.kaggle/kaggle.json

print("📥 Đang tải bộ dữ liệu gốc RSNA từ Kaggle...")
!kaggle datasets download -d kmader/rsna-bone-age -p data/goc --unzip
print("✅ Hoàn tất tải dữ liệu!")

--- (BỎ QUA NẾU BẠN ĐÃ CHỌN CÁCH 1) ---
## CÁCH 2: KẾT NỐI QUA GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Di chuyển vào thư mục dự án trên Drive
%cd /content/drive/MyDrive/gpt-image-bone-age-synthesis

--- (BẮT BUỘC CHO CẢ 2 CÁCH) ---
## BẮT ĐẦU HUẤN LUYỆN LORA

### Bước 3: Cài đặt các thư viện cần thiết

In [ ]:
!pip install pandas transformers diffusers peft accelerate xformers

### Bước 4: Sinh nhãn văn bản lâm sàng (Captions)

In [ ]:
!python models/cau_hinh_B/buoc1_tao_caption.py \
    --csv "data/goc/boneage-training-dataset.csv" \
    --image_dir "data/goc/boneage-training-dataset/boneage-training-dataset"

### Bước 5: Tiến hành huấn luyện LoRA Stable Diffusion Inpainting
* Nếu bạn dùng **Cách 1 (Git)**: Nên chỉnh đường dẫn `--output_dir` sang một thư mục trên Google Drive để tránh bị mất checkpoint khi hết phiên làm việc của Colab.

In [ ]:
# Tạo thư mục trên Drive để lưu checkpoint (nếu dùng Cách 1)
# !mkdir -p /content/drive/MyDrive/lora_out_FULL

!python models/cau_hinh_B/buoc2_train_lora.py \
    --data_dir "data/goc/boneage-training-dataset/boneage-training-dataset" \
    --output_dir "/content/drive/MyDrive/gpt-image-bone-age-synthesis/lora_out_FULL" \
    --num_train_epochs 3 \
    --train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --log_every 20